In [ ]:
%set_env TOKENIZERS_PARALLELISM=false
import os
import numpy as np
import torch

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

from esm.utils.structure.protein_chain import ProteinChain
from esm.utils.constants.esm3 import SEQUENCE_VOCAB
from esm.models.esm3 import ESM3
from esm.sdk.api import (
    ESMProtein,
    GenerationConfig,
)
# model =  ESM3.from_pretrained("esm3_sm_open_v1", device=torch.device(DEVICE)).eval()

# for i, t in enumerate(SEQUENCE_VOCAB):
#     assert model.tokenizer.get_vocab()[t] == i

env: TOKENIZERS_PARALLELISM=false


In [12]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import softmax

import pandas as pd
import numpy as np
import bokeh.plotting
bokeh.io.output_notebook()
from bokeh.models import BasicTicker, PrintfTickFormatter
from bokeh.palettes import viridis, RdBu
from bokeh.transform import linear_cmap
from bokeh.plotting import figure, show

from matplotlib.colors import to_hex
cmap = plt.colormaps["bwr_r"]
bwr_r = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]
cmap = plt.colormaps["gray_r"]
gray = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]

esm_alphabet = SEQUENCE_VOCAB[4:24]
ALPHABET = "AFILVMWYDEKRHNQSTGPC"
ALPHABET_map = [esm_alphabet.index(a) for a in ALPHABET]

Loading BokehJS ...

In [6]:
import math
seq = "MKAKELREKSVEELNTELLNLLREQFNLRMQAASGQLQQSHLLKQVRRDVARVKTLLNEKAGA" # @param {type:"string"}
protein_prompt = ESMProtein(sequence=seq)
protein_tensor = model.encode(protein_prompt)
print(len(seq), protein_tensor.sequence.shape)
k = 2
m = 0.139

protein_prompt = ESMProtein(sequence=seq)
protein_tensor = model.encode(protein_prompt)
x, ln = protein_tensor.sequence, len(seq)
x = x.reshape(1, -1)
res = model(sequence_tokens=x)

# mps = int(ln*m)
# batch_size = math.ceil(ln * k / mps)
# mask_token_idx = torch.tile(torch.reshape(torch.arange(ln)+1, (1, -1)), k+1)

# mask_token_idx = mask_token_idx[torch.randperm(mask_token_idx.shape[0])].reshape(batch_size, mps)
# mask_idx = SEQUENCE_VOCAB.index("<mask>")

# batch_x = torch.tile(x.reshape(1, -1), (batch_size, 1))
# batch_x[torch.arange(batch_size).reshape(-1, 1), mask_token_idx] = mask_idx

63 torch.Size([65])


RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16

In [ ]:
import tqdm.notebook
TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

def get_logits(seq, batch_size=1):
  protein_prompt = ESMProtein(sequence=seq)
  protein_tensor = model.encode(protein_prompt)
  x, ln = protein_tensor.sequence, len(seq)
  with torch.no_grad():
    f = lambda x: model(sequence_tokens=x).sequence_logits[:, 1:(ln+1), 4:24].detach().cpu().numpy()
    logits = np.zeros((ln, 20), dtype=np.float32)
    with tqdm.notebook.tqdm(total=ln, bar_format=TQDM_BAR_FORMAT) as pbar:
      for n in range(0, ln, batch_size):
        m = min(n + batch_size, ln)
        x_h = torch.clone(x).unsqueeze(0).repeat(m - n, 1)
        for i in range(m - n):
          x_h[i, n + i + 1] = SEQUENCE_VOCAB.index("<mask>")
        fx_h = f(x_h.to(DEVICE))
        for i in range(m - n):
          logits[n + i] = fx_h[i, n + i]
        pbar.update(m - n)
  return logits

sequence = "MKAKELREKSVEELNTELLNLLREQFNLRMQAASGQLQQSHLLKQVRRDVARVKTLLNEKAGA" # @param {type:"string"}
# sequence = "MKTFIFLALLGAAVAFPVDDDDKIVGGYTCGANTVPYQVSLNSGYHFCGGSLINSQWVVSAAHCYKSGIQVRLGEDNINVVEGNEQFISASKSIVHPSYNSNTLNNDIMLIKLKSAASLNSRVASISLPTSCASAGTQCLISGWGNTKSSGTSYPDVLKCLKAPILSDSSCKSAYPGQITSNMFCAGYLEGGKDSCQGDSGGPVVCSGKLQGIVSWGSGCAQKNKPGVYTKVCNYVSWIKQTIASN"

sequence = sequence.upper()
sequence = ''.join([i for i in sequence if i.isalpha()])

logits = get_logits(sequence, batch_size=30)
logits = logits[:,ALPHABET_map]

torch.cuda.LongTensor


  0%|          | 0/63 [elapsed: 00:00 remaining: ?]

RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16

In [ ]:
import math
def get_logits_r(seq, device, k=2, m=0.15, batch_size=16):
  protein_prompt = ESMProtein(sequence=seq)
  protein_tensor = model.encode(protein_prompt)
  print(protein_tensor)
  x, ln = protein_tensor.sequence, len(seq)
  mps = int(ln*m)
  num_masked = math.ceil(ln * k / mps)
  mask_token_idx = (torch.arange(mps*num_masked) % ln) + 1
  mask_token_idx = mask_token_idx[torch.randperm(mask_token_idx.shape[0])].reshape(num_masked, mps) #Mut token ind / seq
  mask_idx = SEQUENCE_VOCAB.index("<mask>")

  masked_x = torch.tile(x.reshape(1, -1), (num_masked, 1))
  masked_x[torch.arange(num_masked).reshape(-1, 1), mask_token_idx] = mask_idx
  masked_x.to(device)

  with torch.no_grad():
    logits = torch.zeros((ln, 20))
    for n in range(0, ln, batch_size):
        mn = min(n + batch_size, ln)
        batch_x = x[n:mn]
        batch_mask_idx = mask_token_idx[n:mn]
        batch_logits = model(sequence_tokens=batch_x).sequence_logits[1:(ln+1), 4:24]
        for r in batch_logits.shape[0]:
          logits
        batch_logits

In [9]:
import math
def sample_neighborhood(seq, model, device, mask_frac=0.15, batch_size=16):
    protein_prompt = ESMProtein(sequence=seq)
    protein_tensor = model.encode(protein_prompt)
    x, ln = protein_tensor.sequence, len(seq)
    masked_x.to(device)
    mask_idx = SEQUENCE_VOCAB.index("<mask>")
    num_mask = math.floor(mask_frac*ln)
    mask_token_pos = torch.stack([torch.multinomial(torch.ones(ln) / ln, num_mask) for _ in range(batch_size)]).to(device)
    masked_x = torch.tile(x.reshape(1, -1), (batch_size, 1))
    masked_x[torch.arange(batch_size).reshape(-1, 1), mask_token_pos] = mask_idx
    

In [ ]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
from esm.utils.constants.esm3 import (
    SEQUENCE_MASK_TOKEN,
)
import torch
device = torch.device('cuda:0')
model = ESMC.from_pretrained("esmc_600m").to(device).eval() # or "cpu"

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from go_ml.masking import *
def get_logits(seq, batch_size=8, mask_func=mask_indiv):
    seq_ind = model.encode(ESMProtein(sequence=seq)).sequence
    batch, batch_inds, mut_inds = mask_func(seq_ind, SEQUENCE_MASK_TOKEN)
    bert_eval_l = []
    with torch.no_grad():
        for si in range(0, batch.shape[0], batch_size):
            ei = min(batch.shape[0], si+batch_size)
            x = batch[si:ei, :]
            model_eval = model(x)
            bert_eval = model_eval.sequence_logits
            bert_eval_l.append(bert_eval.cpu())
    bert_eval = torch.cat(bert_eval_l)
    bert_eval = torch.softmax(bert_eval, dim=2)
    bert_mask = (batch == SEQUENCE_MASK_TOKEN).cpu()
    eval_avg, eval_support = mask_avg(bert_mask, bert_eval)
    return eval_avg

sequence = "MKAKELREKSVEELNTELLNLLREQFNLRMQAASGQLQQSHLLKQVRRDVARVKTLLNEKAGA" # @param {type:"string"}
# sequence = "MKTFIFLALLGAAVAFPVDDDDKIVGGYTCGANTVPYQVSLNSGYHFCGGSLINSQWVVSAAHCYKSGIQVRLGEDNINVVEGNEQFISASKSIVHPSYNSNTLNNDIMLIKLKSAASLNSRVASISLPTSCASAGTQCLISGWGNTKSSGTSYPDVLKCLKAPILSDSSCKSAYPGQITSNMFCAGYLEGGKDSCQGDSGGPVVCSGKLQGIVSWGSGCAQKNKPGVYTKVCNYVSWIKQTIASN"

sequence = sequence.upper()
sequence = ''.join([i for i in sequence if i.isalpha()])

logits = get_logits(sequence, batch_size=30, mask_func=mask_indiv)

logits = logits[:,ALPHABET_map]

def pssm_to_dataframe(pssm, esm_alphabet):
  sequence_length = pssm.shape[0]
  idx = [str(i) for i in np.arange(1, sequence_length + 1)]
  df = pd.DataFrame(pssm, index=idx, columns=list(esm_alphabet))
  df = df.stack().reset_index()
  df.columns = ['Position', 'Amino Acid', 'Probability']
  return df

df = pssm_to_dataframe(pssm, ALPHABET)


In [ ]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True



In [ ]:
# pssm = softmax(logits.float(),-1)
pssm = logits.float()

In [23]:
# plot pssm
num_colors = 256  # You can adjust this number
palette = viridis(256)
TOOLS = "hover,save,pan,box_zoom,reset,wheel_zoom"
p = figure(title="CONSERVATION",
           x_range=[str(x) for x in range(1,len(sequence)+1)],
           y_range=list(ALPHABET)[::-1],
           width=900, height=400,
           tools=TOOLS, toolbar_location='below',
           tooltips=[('Position', '@Position'), ('Amino Acid', '@{Amino Acid}'), ('Probability', '@Probability')])

r = p.rect(x="Position", y="Amino Acid", width=1, height=1, source=df,
           fill_color=linear_cmap('Probability', palette, low=0, high=0.1),
           line_color=None)
p.xaxis.visible = False  # Hide the x-axis
show(p)

In [17]:
# plot pssm
num_colors = 256  # You can adjust this number
palette = viridis(256)
TOOLS = "hover,save,pan,box_zoom,reset,wheel_zoom"
p = figure(title="CONSERVATION",
           x_range=[str(x) for x in range(1,len(sequence)+1)],
           y_range=list(ALPHABET)[::-1],
           width=900, height=400,
           tools=TOOLS, toolbar_location='below',
           tooltips=[('Position', '@Position'), ('Amino Acid', '@{Amino Acid}'), ('Probability', '@Probability')])

r = p.rect(x="Position", y="Amino Acid", width=1, height=1, source=df,
           fill_color=linear_cmap('Probability', palette, low=0, high=1),
           line_color=None)
p.xaxis.visible = False  # Hide the x-axis
show(p)

In [4]:
def get_categorical_jacobian(seq, batch_size=1):
  # ∂in/∂out
  protein_prompt = ESMProtein(sequence=seq)
  protein_tensor = model.encode(protein_prompt)
  x, ln = protein_tensor.sequence, len(seq)
  with torch.no_grad():
    f = lambda x: model(sequence_tokens=x).sequence_logits[..., 1:(ln+1), 4:24].detach().cpu().numpy()
    fx = f(x[None].to(DEVICE))[0]
    fx_h = np.zeros([ln, 20, ln, 20], dtype=np.float32)

    with tqdm.notebook.tqdm(total=ln, bar_format=TQDM_BAR_FORMAT) as pbar:
      for n in range(ln):  # for each position
        for i in range(0, 20, batch_size):
          end = min(i + batch_size, 20)
          x_h = torch.clone(x).unsqueeze(0).repeat(end - i, 1).to(DEVICE)
          x_h[:, n+1] = torch.arange(4 + i, 4 + end)
          fx_h[n, i:end] = f(x_h)
        pbar.update(1)

  return fx_h - fx